from previous notebook `06-pre-disambiguation-optimization` we found that the best hyperparameters combinations are:
- scorer: `token_set_ratio`
- score_cutoff: `70`
- processor: `light_normalizer`

we now proceed to make the canonical entities disambiguation checked by an LLM and we give the task to the LLM to give a role to each canonical entity when it is related to a person.

In [ ]:
%load_ext rich

%load_ext autoreload
%autoreload 2

In [ ]:
from __future__ import annotations

import json
import mimetypes
import os
import re
import time
import unicodedata
from collections import Counter
from operator import itemgetter
from pathlib import Path
from typing import Iterable, Tuple

import requests
from tqdm import tqdm

import numpy as np
import pandas as pd
import requests
from more_itertools import flatten, unique_everseen


from aymurai.llm_providers import OllamaLLMProvider
from aymurai.meta.entities import CanonicalEntities, CanonicalEntity
from aymurai.utils.json_data import get_pretty, save_json, load_json
import aymurai.evaluation.metrics as aymurai_metrics
from aymurai.utils.misc import get_element

## /document-extract endpoint output and new endpoint anonymizer/disambiguate

In [ ]:
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
ENDPOINT_DOCUMENT_EXTRACT = f"{API_BASE_URL}/misc/document-extract"

DATA_ROOT = Path(
    os.getenv(
        "DOCUMENT_DATA_ROOT", "../../../resources/data/restricted/disambiguation-eval/files"
    )
)
GOLD_JSON_ROOT = Path(
    os.getenv(
        "GOLD_JSON_ROOT",
        "../../../resources/data/restricted/disambiguation-eval/canonical-entities/manual-predicted",
    )
)

DOC_EXTENSIONS = {".pdf", ".docx"}
JSON_EXTENSION = {".json"}

REQUEST_TIMEOUT_S = float(os.getenv("DOCUMENT_REQUEST_TIMEOUT", "30"))

print(f"Target endpoint: {ENDPOINT_DOCUMENT_EXTRACT}")
print(f"Data root: {DATA_ROOT.resolve()}")

LIMIT = int(os.getenv("DISAMBIGUATION_DOC_LIMIT", "0"))  # 0 = no limit
TARGET_LABELS = os.getenv("DISAMBIGUATION_TARGET_LABELS", "PER,DIRECCION")
TARGET_LABELS = [label.strip() for label in TARGET_LABELS.split(",") if label.strip()]

FUZZY_THRESHOLD = int(os.getenv("DISAMBIGUATION_THRESHOLD", "70"))
FUZZY_SCORER = os.getenv("DISAMBIGUATION_SCORER", "token_set_ratio")
FUZZY_PROCESSOR = os.getenv("DISAMBIGUATION_PROCESSOR", "light_normalizer")

print(f"API: {API_BASE_URL}")
print(f"Data root: {DATA_ROOT}")
print(f"Target labels: {TARGET_LABELS or 'ALL'}")
print(
    f"Fuzzy params: scorer={FUZZY_SCORER}, threshold={FUZZY_THRESHOLD}, processor={FUZZY_PROCESSOR}"
)

In [ ]:
if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Directory '{DATA_ROOT}' not found. Update DATA_ROOT before continuing."
    )


def discover_documents(root: Path, extensions: Iterable[str]) -> list[Path]:
    extensions = {ext.lower() for ext in extensions}
    return sorted(
        path
        for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in extensions
    )


documents = discover_documents(DATA_ROOT, DOC_EXTENSIONS)
if LIMIT > 0:
    documents = documents[:LIMIT]
print(f"Discovered {len(documents)} documents.")

gold_jsons = discover_documents(GOLD_JSON_ROOT, JSON_EXTENSION)
if LIMIT > 0:
    gold_jsons = gold_jsons[:LIMIT]
print(f"Discovered {len(gold_jsons)} gold JSON files.")

## Functions for API endpoints usage

In [ ]:
from aymurai.experiments.entity_disambiguation.runner import (
    call_extraction_api as extract_document,
)

def predict_paragraphs(paragraphs: list[str]) -> list[dict]:
    endpoint = f"{API_BASE_URL}/anonymizer/predict"
    predictions = []
    for paragraph in paragraphs:
        response = requests.post(endpoint, json={"text": paragraph}, timeout=60)
        response.raise_for_status()
        predictions.append(response.json())
    return predictions

def disambiguate(predictions: list[dict]) -> dict:
    endpoint = f"{API_BASE_URL}/anonymizer/disambiguate"
    params = {
        "threshold": FUZZY_THRESHOLD,
        "scorer": FUZZY_SCORER,
        "processor": FUZZY_PROCESSOR,
        "target_labels": TARGET_LABELS,
    }
    if TARGET_LABELS:
        params["target_labels"] = TARGET_LABELS
    response = requests.post(endpoint, params=params, json=predictions, timeout=120)
    response.raise_for_status()
    return response.json()

## Canonical Entity extraction with ***LLM inference***

In [ ]:
# Available models
models = [
    "phi4:14b",
    "gpt-oss:20b",
    "llama3",
    "llama3.1:8b",
    "gemma3:270m",
]

In [ ]:
# Sanity check
provider = OllamaLLMProvider(model=models[0])
provider.generate("hola, ¿cómo estás?")

In [ ]:
system_prompt_1 = """
Eres un asistente experto en análisis legal y anonimización de documentos judiciales.
Tu función es auditar y enriquecer una lista de **Entidades Canónicas (Personas)** pre-agrupadas.

# Material de Trabajo
Para tu análisis, recibirás:
1. **Contexto de la Causa:** Una selección de párrafos del documento original donde se detectaron las menciones.
2. **Entidades Canónicas:** Una lista de grupos de aliases donde cada grupo representa a una única persona física, en teoría.

# Tus Tareas
1. **Validación Semántica:** Usa el contexto para confirmar que los aliases de un grupo realmente refieren a la misma persona física. Si detectas que un grupo contiene personas distintas (ej. homónimos o familiares), sepáralos.
2. **Consolidación:** Si una persona aparece en dos grupos diferentes (ej. "Juan Pérez" en uno y "Juan Alberto Pérez" en otro), únelos en una única entidad canónica.
3. **Asignación de Rol:** Identifica el rol procesal (Juez/a, Denunciante, Imputado/a, Víctima, Abogado/a, etc.) analizando cómo interactúa la persona en los párrafos provistos.

# Reglas de Salida
- Devuelve **únicamente** un objeto JSON que contenga el array de entidades procesadas.
- `canonical_text`: El nombre más completo y formal encontrado.
- `attributes`: Diccionario con el campo `"role"`. Si no se detecta rol, que el diccionario attributes quede vacío. Entre ellos pueden incluirse:
    * "Denunciante"
    * "Denunciado/a"
    * "Víctima"
    * "Juez/a"
    * "Defensor/a de Cámara"
    * "Fiscal"
    * "Abogado/a"
    * "Perito/a"
    * "Testigo"
  Debes asignar un rol tal cual se escribe en esa lista, no inventes nuevos roles.
- Mantén los `aliases` originales tal cual aparecen en el texto.

# Estructura de Respuesta (JSON)
[
  {
    "entity_id": "optional-unique-id",
    "aymurai_label": "PER",
    "canonical_text": "Alias Principal",
    "aliases": [
      "Alias 1",
      "Alias 2"
    ],
    "attributes": {
      "role": "Rol Procesal"
    }
  }
]

# Ejemplo de Input
Contexto: "...la declaración del Sr. Juan Carlos Ruiz ante este Juzgado. El imputado Ruiz negó los cargos. Por su parte, la Dra. Elena Sosa, fiscal de la causa, solicitó..."
Entidades Candidatas:
[{
  "entity_id": "optional-unique-id",
  "aymurai_label": "PER",
  "canonical_text": "Juan Carlos Ruiz",
  "aliases": [
    "Juan Carlos Ruiz",
    "Ruiz"
  ]
},
{
  "entity_id": "optional-unique-id",
  "aymurai_label": "PER",
  "canonical_text": "Elena Sosa",
  "aliases": [
    "Elena Sosa"
  ]
}]

# Ejemplo de Output
[
  {
    "entity_id": "optional-unique-id",
    "aymurai_label": "PER",
    "canonical_text": "Juan Carlos Ruiz",
    "aliases": [
      "Juan Carlos Ruiz",
      "Ruiz"
    ],
    "attributes": {
      "role": "Imputado"
    }
  },
  {
    "entity_id": "optional-unique-id",
    "aymurai_label": "PER",
    "canonical_text": "Elena Sosa",
    "aliases": [
      "Elena Sosa"
    ],
    "attributes": {
      "role": "Fiscal"
    }
  }
]
"""

In [ ]:
system_prompt_2 = """
Eres un asistente experto en análisis legal y anonimización de documentos judiciales.
Tu función es auditar y enriquecer una lista de **Entidades Canónicas (Personas)** pre-agrupadas.

# Material de Trabajo
Para tu análisis, recibirás:
1. **Contexto de la Causa:** Una selección de párrafos del documento original donde se detectaron las menciones.
2. **Entidades Canónicas:** Una lista de grupos de aliases donde cada grupo representa a una única persona física, en teoría.

# Tus Tareas
1. **Validación Semántica:** Usa el contexto para confirmar que los aliases de un grupo realmente refieren a la misma persona física. Si detectas que un grupo contiene personas distintas (ej. homónimos o familiares), sepáralos.
2. **Consolidación:** Si una persona aparece en dos grupos diferentes (ej. "Juan Pérez" en uno y "Juan Alberto Pérez" en otro), únelos en una única entidad canónica.
3. **Asignación de Rol:** Identifica el rol procesal (Juez/a, Denunciante, Imputado/a, Víctima, Abogado/a, etc.) analizando cómo interactúa la persona en los párrafos provistos.

# Reglas de Salida
- Devuelve **únicamente** un objeto JSON que contenga el array de entidades procesadas.
- `canonical_text`: El nombre más completo y formal encontrado.
- `attributes`: Diccionario con el campo `"role"`. Si no se detecta rol, que el diccionario attributes quede vacío. Entre ellos pueden incluirse:
    * "Denunciante"
    * "Denunciado/a"
    * "Víctima"
    * "Juez/a"
    * "Defensor/a de Cámara"
    * "Fiscal"
    * "Abogado/a"
    * "Perito/a"
    * "Testigo"
  Debes asignar un rol tal cual se escribe en esa lista, no inventes nuevos roles.
- Mantén los `aliases` originales tal cual aparecen en el texto.

# Estructura de Respuesta (JSON)
[
  {
    "entity_id": "optional-unique-id",
    "aymurai_label": "PER",
    "canonical_text": "Alias Principal",
    "aliases": [
      "Alias 1",
      "Alias 2"
    ],
    "attributes": {
      "role": "Rol Procesal"
    }
  }
]
"""

In [ ]:
user_prompt_template_1 = """
A continuación se proporcionan fragmentos relevantes de un documento judicial y una lista de entidades (personas) pre-clusterizadas por similitud de texto.

# Contexto de la causa, párrafos con alguna mención a las personas:
{document_text}

# Entidades Pre-clusterizadas a validar:
{canonical_entities}

# Instrucciones:
1. **Validación de Aliases:** Analiza si los aliases dentro de cada entidad canónica pertenecen realmente a la misma persona según los párrafos de contexto. 
   - Si un alias pertenece a una persona distinta (ej. un familiar con nombre similar), sepáralo en una nueva entidad.
   - Si dos grupos de entidades canónicas refieren a la misma persona, fusiónalos.
2. **Identificación de Roles:** Extrae el rol procesal de cada persona basándote en el contexto (ej: Juez/a, Fiscal, Denunciante, Imputado/a, Víctima, Abogado/a, Perito/a, Testigo).
3. **Normalización:** Define el `canonical_text` como el nombre más completo y formal que aparezca en los aliases, si este trabajo ya está bien hecho no lo modifiques.
4. **Formato:** Devuelve la lista final de entidades en el formato JSON solicitado en el system prompt, asegurándote de no omitir a nadie que sea relevante.
"""

In [ ]:
system_prompt_3 = """
Eres un asistente experto en desambiguación de entidades judiciales.
Recibirás fragmentos de una causa y un JSON de entidades canónicas pre-agrupadas.

### Tu Tarea:
1. **Validar y Fusionar:** Verifica que cada entidad represente a una única persona real. Si dos grupos de entidades refieren a la misma persona, fusiónalos.
2. **Limpiar Aliases:** Elimina prefijos (Dr., Sra., Lic., etc.) o sufijos de los nombres (Asesor, Administrativo, etc.). Los aliases deben ser solo nombres propios.
3. **Filtrar:** Elimina cualquier entidad que no sea una persona física.
4. **Asignar Rol:** Clasifica cada entidad según el contexto usando **únicamente** esta lista estandarizada:
   - "Denunciante", "Denunciado/a", "Víctima", "Juez/a", "Defensor/a de Cámara", "Asesor/a Tutelar", "Fiscal", "Abogado/a", "Perito/a", "Testigo".
   - **Nota:** No añadas información extra (ej. "Juez de Cámara" debe ser solo "Juez/a"). Si no se identifica rol, usa `null`.

### Formato de Salida (JSON):
Devuelve exclusivamente un array de objetos:
[
  {
    "entity_id": "optional-unique-id",
    "aymurai_label": "PER",
    "canonical_text": "Alias Principal",
    "aliases": [
      "Alias 1",
      "Alias 2"
    ],
    "attributes": {
      "role": "Rol Procesal"
    }
  }
]
"""

In [ ]:
system_prompt_4 = """
Eres un asistente experto en desambiguación de entidades judiciales.
Recibirás fragmentos de una causa y un JSON de entidades canónicas pre-agrupadas.

### Tu Tarea:
1. **Validar y Fusionar:** Verifica que cada entidad represente a una única persona real. Si dos grupos de entidades refieren a la misma persona, fusiónalos.
2. **Limpiar Aliases:** Elimina prefijos (Dr., Dra., Dres., Sra., Sr., Lic., etc.) o sufijos de los nombres (Asesor, Administrativo, etc.). Los aliases deben ser solo nombres propios.
3. **Filtrar:** Elimina cualquier entidad que no sea una persona física.
4. **Asignar Rol:** Clasifica cada entidad según el contexto usando **únicamente** esta lista estandarizada:
   - "Denunciante", "Denunciado/a", "Víctima", "Juez/a", "Defensor/a de Cámara", "Asesor/a Tutelar", "Fiscal", "Abogado/a", "Perito/a", "Testigo".
   
   **Instrucciones críticas de rol:**
   - Si una persona es mencionada como "Dr.", "Dra." o "Dres." y el contexto no indica que es Juez, Fiscal o Defensor oficial, asígnale el rol de **"Abogado/a"**.
   - No añadas información extra (ej. "Juez de Cámara" debe ser solo "Juez/a"). Si no se identifica un rol de la lista, usa `null`.

### Formato de Salida (JSON):
Devuelve exclusivamente un array de objetos:
[
  {
    "canonical_text": "Nombre Completo Normalizado",
    "aliases": [
      "Alias 1",
      "Alias 2"
    ],
    "attributes": {
      "role": "Rol Procesal"
    }
  }
]
"""

In [ ]:
system_prompt_5 = """
Eres un extractor de roles legales. Tu única función es auditar una lista de entidades pre-agrupadas basándote en fragmentos de texto judicial.

### Instrucciones Estrictas:
1. **Filtrar:** Elimina cualquier entidad que no sea una persona física (ej: calles, instituciones, leyes, fechas).
2. **Asignar Rol:** Identifica el rol procesal usando EXCLUSIVAMENTE esta lista:
   - "Denunciante", "Denunciado/a", "Víctima", "Juez/a", "Defensor/a de Cámara", "Asesor/a Tutelar", "Fiscal", "Abogado/a", "Perito/a", "Testigo".
3. **Regla "Dr/a":** Si la persona es mencionada como "Dr.", "Dra." o "Dres." y no hay otro cargo explícito (como Juez o Fiscal), asígnale "Abogado/a".
4. **Limpiar:** En `canonical_text` y `aliases`, elimina otras palabras que no sean nombres propios.
5. **Fusionar:** Si dos entidades candidatas refieren a la misma persona, devuélvelas como una sola.
"""

In [ ]:
system_prompt_6 = """
Eres un extractor de roles legales. Tu única función es auditar una lista de entidades pre-agrupadas basándote en fragmentos de texto judicial.

### Instrucciones Estrictas:
1. **Filtrar:** Elimina cualquier entidad que no sea una persona física (ej: calles, instituciones, leyes, fechas).
2. **Asignar Rol:** Identifica el rol procesal usando EXCLUSIVAMENTE esta lista:
   - "Denunciante", "Denunciado/a", "Víctima", "Juez/a", "Defensor/a de Cámara", "Asesor/a Tutelar", "Fiscal", "Abogado/a", "Perito/a", "Testigo".
3. **Regla "Dr/a":** Si la persona es mencionada como "Dr.", "Dra." o "Dres." y no hay otro cargo explícito (como Juez o Fiscal), asígnale "Abogado/a".
4. **Fusionar:** Si dos entidades candidatas refieren a la misma persona, devuélvelas como una sola.
5. No modifiques los `canonical_text` ni los `aliases` originales.
"""

In [ ]:
system_prompt_7 = """
Eres un extractor de roles legales. Tu única función es auditar una lista de entidades pre-agrupadas basándote en fragmentos de texto judicial.

### Instrucciones Estrictas:
1. **Filtrar:** Elimina cualquier entidad que no sea una persona física (ej: calles, instituciones, leyes, fechas).
2. **Asignar Rol:** Identifica el rol procesal usando EXCLUSIVAMENTE esta lista:
   - "Denunciante", "Denunciado/a", "Víctima", "Juez/a", "Defensor/a de Cámara", "Asesor/a Tutelar", "Fiscal", "Abogado/a", "Perito/a", "Testigo".
3. **Regla "Dr/a":** Si la persona es mencionada como "Dr.", "Dra." o "Dres." y no hay otro cargo explícito (como Juez o Fiscal), asígnale "Abogado/a".
4. **Manejo de Siglas e Iniciales:** Ten en cuenta que las personas pueden ser mencionadas por sus iniciales (ej: "M.L." para "Martín López"). Si el contexto permite confirmar que unas iniciales refieren a una persona ya identificada, trátalas como un alias de esa misma entidad.
5. **Fusionar:** Si dos entidades candidatas refieren a la misma persona (ya sea por nombre completo, apellido solo o iniciales), devuélvelas como una sola entidad unificada.
6. **Integridad:** No modifiques los `canonical_text` ni los `aliases` originales; mantén la literalidad de lo extraído por el NER.

Devuelve exclusivamente un array de objetos:
[
  {
    "canonical_text": "Nombre Completo Normalizado",
    "aliases": [
      "Alias 1",
      "Alias 2"
    ],
    "attributes": {
      "role": "Rol Procesal"
    }
  }
]
"""

In [ ]:
system_prompt_8 = """
Eres un auditor experto en desambiguación de entidades legales. Tu tarea es validar y enriquecer una lista de personas pre-agrupadas.

### Instrucciones Estrictas:
1. **Validación y Fusión:** - Usa los fragmentos de contexto para confirmar que todos los aliases de un grupo pertenecen a la misma persona.
   - Si detectas que dos entidades distintas son en realidad la misma persona (ej: un grupo con el nombre completo y otro con iniciales o cargo), fusiónalos.
   - Si un alias dentro de un grupo pertenece a una persona diferente, sepáralo.
2. **Filtrado:** Elimina cualquier entidad que no sea una persona física (ej: instituciones, direcciones, leyes).
3. **Manejo de Iniciales:** Identifica si las siglas (ej: "M.L.") corresponden a una persona con nombre completo en el listado y únelas si el contexto lo confirma.
4. **Asignación de Rol:** Identifica el rol procesal basándote EXCLUSIVAMENTE en esta lista:
   - "Denunciante", "Denunciado/a", "Víctima", "Juez/a", "Defensor/a de Cámara", "Asesor/a Tutelar", "Fiscal", "Abogado/a", "Perito/a", "Testigo".
5. **Regla "Dr/a":** Si se menciona como "Dr.", "Dra." o "Dres." y no hay otro cargo explícito, asígnale el rol "Abogado/a".

### Formato de Salida (JSON):
Devuelve un array de objetos con esta estructura:
[
  {
    "entity_id": "optional-unique-id",
    "aymurai_label": "PER",
    "canonical_text": "Nombre de la entidad,
    "aliases": [
      "Nombre de la entidad",
      "Alias 1",
      "Alias 2"
    ],
    "attributes": {
      "role": "Rol de la lista o null"
    }
  }
]
No incluyas el campo 'context' en tu respuesta final.
"""

In [ ]:
system_prompt_9 = """
Eres un auditor experto en desambiguación de entidades legales. Tu tarea es validar y enriquecer una lista de personas pre-agrupadas.

### Instrucciones Estrictas:
1. **Asignación de Rol:** Identifica el rol procesal basándote en las ventanas de contexto dentro de cada entidad canónica en la parte de `attributes`['context']
  Elija un rol EXCLUSIVAMENTE en esta lista:
   - "Denunciante", "Denunciado/a", "Víctima", "Juez/a", "Defensor/a de Cámara", "Asesor/a Tutelar", "Fiscal", "Abogado/a", "Perito/a", "Testigo".
2. **Regla "Dr/a":** Si se menciona como "Dr.", "Dra." o "Dres." y no hay otro cargo explícito, asígnale el rol "Abogado/a".
3. **Validación y Fusión:** - Usa los fragmentos de contexto para confirmar que todos los aliases de un grupo pertenecen a la misma persona.
   - Si detectas que dos entidades distintas son en realidad la misma persona (ej: un grupo con el nombre completo y otro con iniciales o cargo), fusiónalos.
   - Si un alias dentro de un grupo pertenece a una persona diferente, sepáralo.
4. **Filtrado:** Elimina cualquier entidad que no sea una persona física (ej: instituciones, direcciones, leyes).
5. **Limpieza:** Limpiá los `aliases` y el `canonical_text` si los mismos tienen otras palabras, pero respeta que estén escritos de igual manera a que sus ventanas de contexto.
  Te dejo un ejemplo de esta instrucción.
    Vos recibís:
        {
        "canonical_text": "por DREXLER, JORGE",
        "aliases": [
            "por DREXLER, JORGE",
            "Jorge Drexler"
        ],
        "attributes": {
            "context": [
                "Fdo. por DREXLER, JORGE - JUEZ DE CÁMARA el mismo cita en el documento 56 que Juan es culpable",
                "es acaso Jorge Drexler el juez designado para esta causa"
            ]
        }
    
    Debés entregar:
        {
        "canonical_text": "DREXLER, JORGE",
        "aliases": [
            "DREXLER, JORGE",
            "Jorge Drexler"
        ],
        "attributes": {
            "role": "Juez/a"
            ]
        }
        
6. **Manejo de Iniciales:** Identifica si las siglas (ej: "M.L.") corresponden a una persona con nombre completo en el listado y únelas si el contexto lo confirma.

### Formato de Salida (JSON):
Devuelve un array de objetos con esta estructura:
[
  {
    "entity_id": "optional-unique-id",
    "aymurai_label": "PER",
    "canonical_text": "Nombre de la entidad,
    "aliases": [
      "Nombre de la entidad",
      "Alias 1",
      "Alias 2"
    ],
    "attributes": {
      "role": "Rol de la lista o null"
    }
  }
]

No incluyas el campo 'context' en tu respuesta final.
"""

In [ ]:
system_prompt_10 = """
Eres un auditor experto en desambiguación de entidades legales. Tu tarea es validar y enriquecer una lista de personas pre-agrupadas.

### Instrucciones Estrictas:
1. **Validación y Fusión:** - Usa los fragmentos de contexto para confirmar que todos los aliases de un grupo pertenecen a la misma persona.
   - Si detectas que dos entidades distintas son en realidad la misma persona (ej: un grupo con el nombre completo y otro con iniciales o cargo), fusiónalos.
   - Si un alias dentro de un grupo pertenece a una persona diferente, sepáralo.
2. **Filtrado:** Elimina cualquier entidad que no sea una persona física (ej: instituciones, direcciones, leyes).
3. **Manejo de Iniciales:** Identifica si las siglas (ej: "M.L.") corresponden a una persona con nombre completo en el listado y únelas si el contexto lo confirma.
4. **Asignación de Rol:** Identifica el rol procesal basándote EXCLUSIVAMENTE en esta lista:
   - "Denunciante", "Denunciado/a", "Víctima", "Juez/a", "Defensor/a de Cámara", "Asesor/a Tutelar", "Fiscal", "Abogado/a", "Perito/a", "Testigo".
5. **Regla "Dr/a":** Si se menciona como "Dr.", "Dra." o "Dres." y no hay otro cargo explícito, asígnale el rol "Abogado/a".
6. **Limpieza:** No modifiques la literalidad de los `aliases` ni del `canonical_text`.

### Formato de Salida (JSON):
Devuelve un array de objetos con esta estructura:
[
  {
    "entity_id": "optional-unique-id",
    "aymurai_label": "PER",
    "canonical_text": "Nombre de la entidad,
    "aliases": [
      "Nombre de la entidad",
      "Alias 1",
      "Alias 2"
    ],
    "attributes": {
      "role": "Rol de la lista o null"
    }
  }
]
No incluyas el campo 'context' en tu respuesta final.
"""

In [ ]:
system_prompt_11 = """
Eres un auditor experto en desambiguación de entidades legales. Tu tarea es validar y corregir una lista de personas pre-agrupadas basándote en su contexto.

### Instrucciones Estrictas:
1. **Validación de Límites (Boundaries):** El modelo NER a veces incluye palabras adyacentes que no son parte del nombre (ej: "por", "Fdo.", "Dres", "dijo", "con"). Tu tarea es corregir los `aliases` y el `canonical_text` para que contengan ÚNICAMENTE el nombre de la persona, pero manteniendo la literalidad exacta (mayúsculas, tildes, comas) de cómo está escrito en el fragmento de `context`.
2. **Fusión y Desambiguación:** - Si dos grupos de entidades refieren a la misma persona física, fusiónalos.
   - Si una persona aparece por sus iniciales (ej: "M.L."), asóciala como alias del nombre completo si el contexto lo confirma.
3. **Filtrado:** Elimina entidades que no sean personas físicas.
4. **Asignación de Rol:** Usa exclusivamente: "Denunciante", "Denunciado/a", "Víctima", "Juez/a", "Defensor/a de Cámara", "Asesor/a Tutelar", "Fiscal", "Abogado/a", "Perito/a", "Testigo".
5. **Regla "Dr/a":** Si se menciona como "Dr.", "Dra." o "Dres." y no hay otro cargo explícito, el rol es "Abogado/a".
6. **Limpieza:** No modifiques la literalidad de los `aliases` ni del `canonical_text`.

### Formato de Salida (JSON):
[
  {
    "entity_id": "ID original",
    "aymurai_label": "PER",
    "canonical_text": "Nombre exacto extraído del texto",
    "aliases": [
      "Nombre exacto extraído del texto (Igual al canonical)",
      "Otros aliases exactos del texto..."
    ],
    "attributes": {
      "role": "Rol de la lista o null"
    }
  }
]
**IMPORTANTE:** El primer elemento de `aliases` debe ser idéntico al `canonical_text`. No incluyas el campo 'context' en la respuesta.
"""

In [ ]:
system_prompt_DIR = """
Eres un auditor experto en desambiguación de domicilios y direcciones en documentos judiciales. Tu tarea es validar y consolidar una lista de direcciones pre-agrupadas basándote en su contexto.

### Instrucciones Estrictas:
1. **Consolidación Geográfica:** - Si dos entidades refieren a la misma ubicación física, fusiónalas en una sola entidad. 
   - Considera que una dirección puede estar escrita de forma completa en un alias (ej: "Av. Rivadavia 123, CABA") y de forma abreviada en otro (ej: "Rivadavia 123"). 
   - Trata las variantes ortográficas o abreviaturas (Av. vs Avenida, Piso vs P., Dpto vs Departamento) como aliases de la misma ubicación.
2. **Filtrado:** Elimina entidades que no sean direcciones postales o domicilios físicos (ej: correos electrónicos, URLs, nombres de juzgados o menciones abstractas como "el domicilio constituido").

### Formato de Salida (JSON):
[
  {
    "entity_id": "ID original",
    "aymurai_label": "DIRECCION",
    "canonical_text": "Dirección más completa extraída del texto",
    "aliases": [
      "Dirección más completa (Igual al canonical)",
      "Variante 1",
      "Variante 2"
    ],
    "attributes": []
  }
]
**IMPORTANTE:** No incluyas el campo 'context' en la respuesta.
"""

In [ ]:
user_prompt_template_DIR= """
Se proporciona la lista de entidades (direcciones) pre-clusterizadas. Cada una incluye en `attributes['context']` las ventanas de texto de ±120 caracteres donde fue mencionada en la causa judicial para que puedas determinar su validez y si hay otras entidades de direcciones que corresponden a esa.

# Entidades Pre-clusterizadas a validar:
{canonical_entities}

Procesa la lista siguiendo las instrucciones de filtrado y fusión.
"""

In [ ]:
user_prompt_template_2 = """
A continuación se proporcionan fragmentos relevantes de un documento judicial y una lista de entidades (personas) pre-clusterizadas por similitud de texto.

# Contexto de la causa, párrafos con alguna mención a las personas:
{document_text}

# Entidades Pre-clusterizadas a validar:
{canonical_entities}
"""

In [ ]:
user_prompt_template_3 = """
Se proporciona la lista de entidades (personas) pre-clusterizadas. Cada una incluye en `attributes['context']` las ventanas de texto de ±120 caracteres donde fue mencionada en la causa judicial para que puedas determinar su rol y validez.

# Entidades Pre-clusterizadas a validar:
{canonical_entities}

Procesa la lista siguiendo las instrucciones de filtrado, fusión y asignación de roles.
"""

In [ ]:
import copy

def add_canonical_entities_context(
    predictions: list[dict], 
    entities: list[dict],
    context_window_length: int | None = 120,
    target_label: str | None = None
) -> list[dict]:
    """
    Creates a deep copy of entities and adds context. 
    
    Args:
        predictions: List of prediction dictionaries from the API.
        entities: List of canonical entities.
        context_window_length: Length of the window around the label. If None, full paragraph is used.
        target_label: The specific label to filter. If None, all labels are considered.
    """
    # 1. Deep copy to protect original variable
    entities_with_context = copy.deepcopy(entities)

    # 2. Process each entity
    for entity in entities_with_context:
        if 'attributes' not in entity or entity['attributes'] is None:
            entity['attributes'] = {}
        if 'context' not in entity['attributes']:
            entity['attributes']['context'] = []
            
        context_windows = set()
        aliases = [a for a in entity.get('aliases', [])]
        
        # If target_label is not provided as an argument, 
        # we can default to the entity's own label if it exists
        current_target = target_label or entity.get('aymurai_label')

        # 3. Iterate through predictions
        for pred in predictions:
            doc_text = pred.get('document', '')
            labels = pred.get('labels', [])
            
            for label in labels:
                label_attr = label['attrs'].get('aymurai_label')
                label_text = label.get('text', '')
                
                # Logic Gate: Filter by label type if current_target is specified
                if current_target is None or label_attr == current_target:
                    
                    # Logic Gate: Check if any alias is inside the label text
                    if any(alias in label_text for alias in aliases):
                        
                        # Handle Window vs Full Paragraph
                        if context_window_length is None:
                            snippet = doc_text
                        else:
                            start = label['start_char']
                            end = label['end_char']
                            
                            window_start = max(0, start - context_window_length)
                            window_end = min(len(doc_text), end + context_window_length)
                            snippet = doc_text[window_start:window_end]

                        clean_snippet = " ".join(snippet.split())
                        context_windows.add(clean_snippet)

        # Update entity with the collected windows
        entity['attributes']['context'] = list(context_windows)

    return entities_with_context

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-4")

def get_phi4_tokens(system_prompt, user_prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    full_text = tokenizer.apply_chat_template(messages, tokenize=False)
    tokens = tokenizer.encode(full_text)
    
    return len(tokens)

def llm_infer_canonical_entities(
    system_prompt: str,
    user_prompt_template: str,
    model: str,
    document_path: Path,
    gold_path: Path,
    context_window_length: int | None = 120,
    model_context: int = 9_500,
    token_limit_frac: float = 2/3,
    target_label: str = "PER",
    temperature: int = 0,
    decompose_by: int | None = 0,
) -> dict:
    
    """
    Infers canonical entities using an LLM by providing context windows 
    around detected PER entities and pre-clusterization results.
    """

    # 1. Filename cleaning
    clean_base_name = document_path.stem.replace("_", "-").replace(" ", "-")
    target_filename = re.sub(r"-{2,}", "-", clean_base_name).strip("-")

    print(f"----- Processing document: {target_filename} -----")

    # 2. Extract document text via API
    session = requests.Session()
    api_response = extract_document(
        session,
        file_path = Path(document_path),
        endpoint=f"{API_BASE_URL}/misc/document-extract",
        timeout_s=300,
    )
    
    document_paragraphs = api_response.get("detail", {}).get("document")

    if not document_paragraphs:
        raise ValueError("Document text is empty or not found for {target_filename}.")

    # 3. Get the pre-clustered entities with context from API calls
    predictions = predict_paragraphs(document_paragraphs)

    disambiguated = disambiguate(predictions)

    canonical_entities_pre_cluster = disambiguated.get("canonical_entities", [])

    canonical_entities_with_context = add_canonical_entities_context(
        predictions=predictions,
        entities=canonical_entities_pre_cluster,
        context_window_length=context_window_length,
        target_label=target_label
    )

    # 4. Clean entities to save tokens
    canonical_entities_prompt = [
        {
            k: v
            for k, v in ce.items()
            if k in ("canonical_text","aliases","attributes")
        }
        for ce in canonical_entities_with_context
    ]

    token_limit = int(model_context*token_limit_frac)

    # 5. Find Optimal Batch Size if decompose_by is None
    if decompose_by is None:
        current_batch_size = len(canonical_entities_prompt)
        print(f"{'-' * 60}")
        print("Searching for optimal batch size...")
        print(f"Starting with {current_batch_size} entities.")
        
        while current_batch_size > 0:
            # Test with the first N entities
            test_batch = canonical_entities_prompt[:current_batch_size]
            test_prompt = user_prompt_template.format(
                canonical_entities=get_pretty(test_batch)
            )
            
            num_tokens = get_phi4_tokens(system_prompt, test_prompt)
            
            if num_tokens <= token_limit:
                decompose_by = current_batch_size
                print(f"{'-' * 60}")
                print(f"Optimal batch size found: {decompose_by} entities ({num_tokens} tokens).")
                break
            
            current_batch_size -= 1
        
        if not decompose_by: # Safety check if even 1 entity is too large
            print("\nEven a single entity exceeds the token limit.")
            return None

    # 6. Prepare batches. If decompose_by is 0 or less, we put all entities in one single list (one batch)
    if decompose_by <= 0:
        entity_batches = [canonical_entities_prompt]
    else:
        entity_batches = [
            canonical_entities_prompt[i : i + decompose_by]
            for i in range(0, len(canonical_entities_prompt), decompose_by)
        ]

    all_raw_outputs = []
    all_user_prompts = []

    # 7. Iterate through batches
    for batch_index, batch in enumerate(entity_batches):
        print(f"{'-' * 60}")
        print(f"Processing batch {batch_index + 1}/{len(entity_batches)}.")
        
        user_prompt = user_prompt_template.format(
            canonical_entities=get_pretty(batch),
        )
        
        all_user_prompts.append(user_prompt)

        # 7.A Token Validation
        len_tokens = get_phi4_tokens(system_prompt, user_prompt)
        if len_tokens > model_context:
            print(f"Batch {batch_index} too large ({len_tokens} tokens).")
            print("Skipping batch.")
            continue
        else:
            print(f"Total batch tokens: {len_tokens} within model context ({model_context}).")

        # 7.B LLM Inference
        provider = OllamaLLMProvider(model=model)
        response = provider.generate(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            options={"temperature": temperature, "num_ctx": model_context},
            format=CanonicalEntities.model_json_schema(),
        )

        # 7.C Parse and collect results
        batch_entities = json.loads(response.text).get("canonical_entities", [])
        all_raw_outputs.extend(batch_entities)


    canonical_entities_llm = [
        CanonicalEntity.model_validate(canonical_entity)
        for canonical_entity in all_raw_outputs
    ]

    canonical_entities_llm = [
        entity.model_dump() | {"entity_id": entity.entity_id.hex}
        for entity in canonical_entities_llm
    ]

    # 8. Load Gold Standard for reference
    canonical_entities_gold = load_json(json_file_path=gold_path)

    print(f"{'-' * 60}")
    print(f"Summary for {target_filename}:")
    print(f"- Pre-cluster entities: {len(canonical_entities_pre_cluster)}")
    print(f"- LLM generated entities: {len(canonical_entities_llm)}")
    print(f"- Gold standard entities: {len(canonical_entities_gold)}")

    # 9. Map the canonical entities in the predictions documents te return the right format for the front-end

    predictions_llm = copy.deepcopy(predictions)
    
    for document in predictions_llm:
        if document.get('labels'):
            labels = document.get('labels','')

            for label in labels:
                label_text = label.get('text', '')
                if label['attrs']['canonical_entity_id'] is None:
                    for ce in canonical_entities_llm:
                        entity_id = ce.get('entity_id')
                        attributes = ce.get('attributes')
                        role = attributes.get('role')
                        aliases = ce.get('aliases')

                        if any(alias in label_text for alias in aliases):
                            label['attrs']['canonical_entity_id'] = entity_id
                            label['attrs']['aymurai_label_subclass'].append(role)

    # 10. Return structured results
    return {
        "target_filename": target_filename,
        "pre_cluster_json": canonical_entities_pre_cluster,
        "llm_output_json": canonical_entities_llm,
        "gold_standard_json": canonical_entities_gold,
        "system_prompt": system_prompt,
        "user_prompts": all_user_prompts,
        "len_tokens": len_tokens,
        "predictions": predictions_llm
    }

In [ ]:
def get_document_paths(document_check: Path, pre_clusterization_root: Path) -> dict:
    """
    Generates the pre-cluster and gold JSON paths for a given document.
    
    Args:
        document_check (Path): The original document Path object.
        pre_clusterization_root (Path): The root directory for pre-clusterization data.
        
    Returns:
        dict: A dictionary containing the 'pre_cluster_path' and 'gold_path'.
    """
    # 1. Prepare target filename
    clean_base_name = document_check.stem.replace("_", "-").replace(" ", "-")
    target_filename = re.sub(r"-{2,}", "-", clean_base_name).strip("-")

    # 2. Define subdirectories
    pre_cluster_dir = pre_clusterization_root / 'best-pre-clusterization-per'
    gold_dir = pre_clusterization_root / 'gold-jsons'

    # 3. Build final paths
    pre_cluster_json_path = pre_cluster_dir / f"{target_filename}-ner-preds-pre-cluster.json"
    gold_json_path = gold_dir / f"{target_filename}-canonical-entities-gold.json"

    return {
        "target_filename": target_filename,
        "pre_cluster_path": pre_cluster_json_path,
        "gold_path": gold_json_path
    }

In [ ]:
PRE_CLUSTERIZATION_ROOT = GOLD_JSON_ROOT.parent / "pre-clusterization"

In [ ]:
import shutil

def run_llm_grid_search(
    models: list,
    system_prompts: dict,
    user_prompt_templates: dict,
    context_window_lengths: list,
    model_context_lengths: list,
    temperatures: list,
    token_limit_frac: float,
    documents: list[Path],
    target_label: str,
    base_output_path: Path,
    decompose_by: int | None = 0
):
    """
    Runs a grid search over hyperparameters using nested loops.
    """

    # To keep track of all results and find the best
    all_combinations_results = []

    for model in models:
        for system_prompt_name, system_prompt in system_prompts.items():
            for user_prompt_template_name, user_prompt_template in user_prompt_templates.items():
                for context_window_length in context_window_lengths:
                    for model_context in model_context_lengths:
                        for temperature in temperatures:
                        
                            print(
                                f"\nRunning grid search with model: {model}, "
                                f"context_window_length: {context_window_length}, "
                                f"model_context: {model_context}, "
                                f"temperature: {temperature}"
                            )

                            # 1. Create a descriptive folder name with the cleaned names
                            combo_name = (
                                f"model-{model}-{system_prompt_name}-"
                                f"{user_prompt_template_name}-context_window-{context_window_length}-"
                                f"model_context-{model_context}-temperature-{temperature}"
                            )

                            combo_dir = base_output_path / combo_name
                            combo_dir.mkdir(parents=True, exist_ok=True)

                            metrics_results = {}
                            metrics_file_path = (
                                combo_dir / f"evaluation_metrics_{target_label}.json"
                            )

                            # List to collect scores for this specific combination
                            current_combo_scores = []

                            for document in documents:
                                
                                # Get document paths
                                paths = get_document_paths(document, PRE_CLUSTERIZATION_ROOT) # We use this because the gold_json is in an specific folder

                                # Run LLM inference
                                llm_response_dict = llm_infer_canonical_entities(
                                    system_prompt=system_prompt,
                                    user_prompt_template=user_prompt_template,
                                    model=model,
                                    document_path=document,
                                    gold_path=paths['gold_path'], # When adding this to the experiment runner, make sure to pass the correct gold path
                                    context_window_length=context_window_length,
                                    model_context=model_context,
                                    target_label=target_label,
                                    temperature=temperature,
                                    decompose_by=decompose_by,
                                    token_limit_frac=token_limit_frac
                                )
                                # Extract predictions
                                ce_llm = llm_response_dict['llm_output_json']
                                
                                # Validate and prepare for saving
                                ce_llm = [
                                    CanonicalEntity.model_validate(entity)
                                    for entity in ce_llm
                                ]

                                ce_llm = [
                                    entity.model_dump() | {"entity_id": entity.entity_id.hex}
                                    for entity in ce_llm
                                ]

                                # Save the result as the llm-pred.json
                                target_filename = llm_response_dict['target_filename']
                                llm_filename = f"{target_filename}-llm-pred.json"
                                llm_output_path = combo_dir / llm_filename

                                save_json(
                                    file_path=llm_output_path, json_data=ce_llm
                                )

                                # Extract gold standard
                                ce_gold = llm_response_dict['gold_standard_json']
                                
                                # Validate and prepare for saving
                                ce_gold = [
                                    CanonicalEntity.model_validate(entity)
                                    for entity in ce_gold
                                ]

                                ce_gold = [
                                    entity.model_dump() | {"entity_id": entity.entity_id.hex}
                                    for entity in ce_gold
                                ]

                                len_llm = len(ce_llm)
                                len_gold = len(ce_gold)

                                # Evaluate metrics
                                score, metrics = aymurai_metrics.evaluate_disambiguation(
                                    gold_json=ce_gold,
                                    pred_json=ce_llm,
                                    target_label=target_label,
                                )
                                current_combo_scores.append(score)
                                doc_id = target_filename
                                metrics_results[doc_id] = {
                                    "label": target_label,
                                    "metric_value": score,
                                    "detailed_metrics": metrics,
                                    "len_llm_predictions": len_llm,
                                    "len_gold_standard": len_gold
                                }

                            # --- Calculate Average for this combination ---
                            avg_score = (
                                np.mean(current_combo_scores) if current_combo_scores else 0.0
                            )

                            # Add average to the results file for this folder
                            final_output = {
                                "average_combination_score": avg_score,
                                "document_details": metrics_results,
                            }

                            with open(metrics_file_path, "w") as f:
                                json.dump(final_output, f, indent=4)

                            # Store combination info for final ranking
                            all_combinations_results.append(
                                {
                                    "name": combo_name,
                                    "score": avg_score,
                                    "params": {
                                        "model": model,
                                        "system_prompt": system_prompt_name,
                                        "user_prompt_template": user_prompt_template_name,
                                        "context_window_length": context_window_length,
                                        "model_context": model_context
                                    },
                                }
                            )

    # Save all combinations results in a summary file
    summary_file_path = (
        base_output_path / f"results_summary_{target_label}.json"
    )
    with open(summary_file_path, "w") as f:
        json.dump(all_combinations_results, f, indent=4)

    # --- Find the best combination ---
    best_combo = max(all_combinations_results, key=lambda x: x["score"])

    print(f"\nGrid Search for {target_label} completed.")

    # We now copy the best combination folder to a new location with a standardized name
    src = Path(base_output_path) / best_combo["name"]
    # Combine the parent destination path with the new folder name
    new_name = f"best-llm-pred-{target_label.lower()}"
    dest = Path(base_output_path.parent) / new_name
    
    try:
        # copytree creates the destination directory with the 'new_name'
        shutil.copytree(src, dest)
        print(f"Successfully copied '{src.name}' to '{dest}'")
    except FileExistsError:
        print(f"Error: A folder named '{new_name}' already exists in '{base_output_path.parent}'")
    except Exception as e:
        print(f"An error occurred: {e}")    

    return best_combo

In [ ]:
models = [
    "phi4:14b",
    "gpt-oss:20b",
    "llama3",
    "llama3.1:8b",
    "gemma3:270m",
]

system_prompts = {
    "system_prompt_1": system_prompt_1,
    "system_prompt_2": system_prompt_2,
    "system_prompt_3": system_prompt_3,
    "system_prompt_4": system_prompt_4,
    "system_prompt_5": system_prompt_5,
    "system_prompt_6": system_prompt_6,
    "system_prompt_7": system_prompt_7,
    "system_prompt_8": system_prompt_8,
    "system_prompt_9": system_prompt_9,
    "system_prompt_10": system_prompt_10,
    "system_prompt_11": system_prompt_11
}


user_prompt_templates = {
    "user_prompt_template_1": user_prompt_template_1,
    "user_prompt_template_2": user_prompt_template_2,
    "user_prompt_template_3": user_prompt_template_3,
}

context_window_lengths = list(range(80, 200, 10))

model_context_lengths = [7_500, 8_000, 8_500, 9_000, 9_500]

temperatures = list(np.arange(0,0.3,0.1))

In [ ]:
# tokens_doc = {}

# for document in documents:
#     # 2. Extract document text via API
#     session = requests.Session()
#     api_response = extract_document(
#         session,
#         file_path = Path(document),
#         endpoint=f"{API_BASE_URL}/misc/document-extract",
#         timeout_s=300,
#     )

#     document_paragraphs = api_response.get("detail", {}).get("document")

#     if not document_paragraphs:
#         raise ValueError("Document text is empty or not found for {target_filename}.")

#     # 3. Get the pre-clustered entities with context from API calls
#     predictions = predict_paragraphs(document_paragraphs)

#     disambiguated = disambiguate(predictions)

#     canonical_entities_pre_cluster = disambiguated.get("canonical_entities", [])

#     canonical_entities_with_context = add_canonical_entities_context(
#         predictions=predictions,
#         entities=canonical_entities_pre_cluster,
#         context_window_length=120,
#         target_label='PER'
#     )

#     # 4. Clean entities to save tokens
#     canonical_entities_prompt = [
#         {
#             k: v
#             for k, v in ce.items()
#             if k in ("canonical_text","aliases","attributes")
#         }
#         for ce in canonical_entities_with_context
#     ]

#     # 5. Build Final User Prompt
#     user_prompt = user_prompt_templates['user_prompt_template_3'].format(
#         canonical_entities=get_pretty(canonical_entities_prompt),
#     )

#     total = get_phi4_tokens(system_prompts['system_prompt_8'], user_prompt)
#     tokens_doc[document.stem] = {
#         "tokens": total,
#         "path": document
#         }
    
#     print(f"Tokens para Phi-4 del documento {document.stem}: {total}")

In [ ]:
document_check = documents[7]
paths = get_document_paths(document_check, PRE_CLUSTERIZATION_ROOT)

In [ ]:
llm_response_dict = llm_infer_canonical_entities(
    system_prompt=system_prompts['system_prompt_9'],
    user_prompt_template=user_prompt_templates['user_prompt_template_3'],
    model=models[0],
    document_path=document_check,
    gold_path=paths["gold_path"],
    context_window_length=120,
    model_context=9_500,
    token_limit_frac=2/3,
    temperature=0,
    target_label="PER",
    decompose_by=None
)

In [ ]:
llm_response_dict = llm_infer_canonical_entities(
    system_prompt=system_prompt_DIR,
    user_prompt_template=user_prompt_template_DIR,
    model=models[0],
    document_path=document_check,
    gold_path=paths["gold_path"],
    context_window_length=120,
    model_context=9_500,
    token_limit_frac=2/3,
    temperature=0,
    target_label="DIRECCION",
    decompose_by=None
)

In [ ]:
llm_response_dict['predictions']

In [ ]:
llm_response_dict['llm_output_json']

In [ ]:
llm_response_dict["gold_standard_json"]

In [ ]:
llm_response_dict["pre_cluster_json"]

In [ ]:
print(llm_response_dict["user_prompts"][2])

In [ ]:
print(llm_response_dict['system_prompt'])

In [ ]:
models = [
    "phi4:14b",
]

system_prompts = {
    "system_prompt_8": system_prompt_8
}


user_prompt_templates = {
    "user_prompt_template_3": user_prompt_template_3,
}

context_window_lengths = [120]

model_context_lengths = [9_500]

temperatures = [0]

In [ ]:
able_documents = []
for k,v in tokens_doc.items():
    if v['tokens'] < 7000:
        able_documents.append(v['path'])


In [ ]:
able_documents

In [ ]:
len(able_documents)

In [ ]:
top_result = run_llm_grid_search(
    models=models,
    system_prompts=system_prompts,
    user_prompt_templates=user_prompt_templates,
    context_window_lengths=context_window_lengths,
    model_context_lengths=model_context_lengths,
    temperatures=temperatures,
    documents=documents,
    target_label="PER",
    base_output_path=GOLD_JSON_ROOT.parent / "llm-grid-search-results",
    token_limit_frac=2/3,
    decompose_by=None
)

In [ ]:
print(
    f"The best hyperparameter combination is '{top_result['name']}' "
    f"with an average score of {top_result['score']:.4f}."
)